# Download dos Dados do S&P 500

Este notebook realiza:

1. Coleta dos componentes atuais do índice S&P 500;
2. Armazenamento da lista de tickers utilizada no projeto;
3. Download do histórico de preços via Yahoo Finance;
4. Armazenamento dos dados brutos para as próximas etapas do pipeline.

In [9]:
# Bibliotecas utilizadas

import pandas as pd
import yfinance as yf

from pathlib import Path

## Configuração do Projeto

Nesta etapa são definidos os parâmetros gerais utilizados ao longo do notebook:

- Índice analisado;
- Período histórico dos dados;
- Nomes dos arquivos gerados;
- Diretório de armazenamento dos dados brutos.

In [10]:
START_DATE = "2018-01-01"
INDEX_NAME = "S&P 500"
TICKERS_FILE = "sp500_tickers.csv"
PRICES_FILE = "sp500_prices_raw.csv"

# Diretório onde os dados brutos serão armazenados
DATA_DIR = Path("../data/raw")

# Cria o diretório caso ele não exista
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Obtenção dos componentes do S&P 500

A lista de ativos é obtida a partir da página pública da Wikipédia
que contém a composição atual do índice S&P 500.

A lista será salva localmente para garantir reprodutibilidade dos
experimentos.

In [11]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(f"Status da requisição: {response.status_code}")

html = StringIO(response.text)

sp500 = pd.read_html(html)[0]

sp500.head()

Status da requisição: 200


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


## Preparação dos tickers

Alguns ativos possuem ponto no ticker (por exemplo: BRK.B).

O Yahoo Finance utiliza hífen nesses casos (BRK-B), portanto
realizamos a conversão.

In [12]:
# Extração dos tickers

tickers = sp500["Symbol"].tolist()

# Ajuste de compatibilidade com o Yahoo Finance

tickers = [ticker.replace(".", "-") for ticker in tickers]

print(f"Índice selecionado: {INDEX_NAME}")
print(f"Quantidade de ativos encontrados: {len(tickers)}")
print()
print("Primeiros 10 ativos:")
print(tickers[:10])

Índice selecionado: S&P 500
Quantidade de ativos encontrados: 503

Primeiros 10 ativos:
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## Salvando a lista de ativos

A lista de ativos utilizada será armazenada localmente para evitar
dependência de fontes externas durante as próximas etapas do projeto.

In [13]:
# Criação do dataframe contendo os tickers

sp500_tickers = sp500.copy()
sp500_tickers["YahooSymbol"] = tickers

sp500_tickers.to_csv(
    DATA_DIR / TICKERS_FILE,
    index=False
)

sp500.to_csv(
    DATA_DIR / "sp500_companies.csv",
    index=False
)

print("Arquivo salvo com sucesso.")

Arquivo salvo com sucesso.


## Download dos preços históricos

Nesta etapa são obtidos os dados históricos de todos os ativos do
S&P 500 utilizando a biblioteca yfinance.

O período analisado será de 2018 até a data atual.

In [14]:
# Download dos dados históricos

prices = yf.download(
    tickers=tickers,
    start=START_DATE,
    auto_adjust=True,
    group_by="ticker",
    threads=True
)

[*********************100%***********************]  503 of 503 completed


## Inspeção inicial dos dados

In [15]:
# Verificação de ativos sem dados de fechamento

failed_tickers = []

for ticker in tickers:
    if ticker not in prices.columns.get_level_values(0):
        failed_tickers.append(ticker)
    elif prices[ticker]["Close"].dropna().empty:
        failed_tickers.append(ticker)

print(f"Ativos com problema no download: {len(failed_tickers)}")
print(failed_tickers[:20])

Ativos com problema no download: 0
[]


In [16]:
# Dimensão do conjunto de dados

print(prices.shape)

# Visualização das primeiras linhas

prices.head()

(2123, 2515)


Ticker             AZO                                              \
Price             Open        High         Low       Close  Volume   
Date                                                                 
2018-01-02  716.539978  746.539978  714.450012  736.539978  558700   
2018-01-03  740.130005  760.000000  736.760010  749.429993  597400   
2018-01-04  759.890015  769.179993  751.219971  761.260010  641000   
2018-01-05  767.380005  780.539978  764.719971  775.500000  483000   
2018-01-08  775.159973  782.429993  761.780029  766.479980  430900   

Ticker             URI                                               ...  \
Price             Open        High         Low       Close   Volume  ...   
Date                                                                 ...   
2018-01-02  166.752787  167.755704  165.576297  167.389252   813300  ...   
2018-01-03  168.228220  169.472221  167.215666  168.941833   844100  ...   
2018-01-04  169.578296  170.320823  163.580132  165.865601  1331000  ...   
2018-01-05  166.829964  167.138558  163.454791  165.267746  1078600  ...   
2018-01-08  165.180883  166.511670  163.387220  166.299515  1623600  ...   

Ticker            HAL                                                   XOM  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2018-01-02  41.302977  42.079732  41.134121  41.885544   7269200  57.357491   
2018-01-03  42.045981  43.126683  41.809579  42.636990  11181900  58.274443   
2018-01-04  42.940936  43.717688  42.409029  43.591045  10142700  59.389857   
2018-01-05  43.447506  43.818999  43.016916  43.751453   8617300  59.362461   
2018-01-08  43.616366  44.207374  43.396850  44.148273   7611900  59.328262   

Ticker                                                 
Price            High        Low      Close    Volume  
Date                                                   
2018-01-02  58.301815  57.248007  58.185486  11469300  
2018-01-03  59.513013  58.041781  59.328251  13957700  
2018-01-04  59.684104  59.143511  59.410385  10863000  
2018-01-05  59.451417  58.650795  59.362461  11047600  
2018-01-08  59.636197  59.259833  59.629353  10927100  

[5 rows x 2515 columns]

In [22]:
prices.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2123 entries, 2018-01-02 to 2026-06-12
Columns: 2515 entries, ('AZO', 'Open') to ('XOM', 'Volume')
dtypes: float64(2040), int64(475)
memory usage: 40.8 MB


In [21]:
print(f"Quantidade de ativos: {len(tickers)}")
print(f"Quantidade de pregões: {len(prices)}")

Quantidade de ativos: 503
Quantidade de pregões: 2123


## Salvando os dados brutos

Os dados serão utilizados posteriormente na etapa de engenharia
de atributos.

In [17]:
# Salvando os preços históricos

prices.to_csv(
    DATA_DIR / PRICES_FILE
)

print("Arquivo salvo com sucesso.")

Arquivo salvo com sucesso.


## Próximos Passos

No próximo notebook serão realizadas as etapas de:

- Limpeza dos dados;
- Tratamento de valores ausentes;
- Criação das variáveis derivadas;
- Construção da variável alvo (retorno futuro de 21 pregões).